# Project 3: Name Gender Classifier

**Sabina Baraili**

This notebook follows Exercise 6.10.2 from *Natural Language Processing with Python*. I start with the book's baseline name-gender classifier, make incremental feature improvements, use the dev-test set to compare models, and then report final performance on the held-out test set.

## Imports and Setup

I use a fixed random seed so the split is reproducible if the notebook is run again.

In [1]:
import random
import subprocess
import sys

try:
    import nltk
except ModuleNotFoundError:
    subprocess.check_call([
        sys.executable,
        '-m',
        'pip',
        'install',
        '--user',
        '--break-system-packages',
        '--quiet',
        'nltk',
    ])
    import nltk

from nltk.classify import apply_features
from nltk.classify.maxent import MaxentClassifier
from nltk.classify.naivebayes import NaiveBayesClassifier
from nltk.corpus import names

SEED = 42
random.seed(SEED)

In [2]:
nltk.download('names', quiet=True)

True

## Load and Split the Corpus

The assignment asks for 500 names in the test set, 500 in the dev-test set, and the remaining names for training.

In [3]:
male_names = [(name, 'male') for name in names.words('male.txt')]
female_names = [(name, 'female') for name in names.words('female.txt')]
all_names = male_names + female_names
random.shuffle(all_names)

test_set = all_names[:500]
devtest_set = all_names[500:1000]
train_set = all_names[1000:]

print(f"Total names: {len(all_names)}")
print(f"Training set size: {len(train_set)}")
print(f"Dev-test set size: {len(devtest_set)}")
print(f"Test set size: {len(test_set)}")

Total names: 7944
Training set size: 6944
Dev-test set size: 500
Test set size: 500


## Baseline Features

The book's example uses the last letter of the name, so I begin there.

In [4]:
def gender_features_basic(name):
    name = name.lower()
    return {'last_letter': name[-1]}


print(gender_features_basic('John'))
print(gender_features_basic('Mary'))

{'last_letter': 'n'}
{'last_letter': 'y'}


In [5]:
train_features_basic = apply_features(gender_features_basic, train_set)
devtest_features_basic = apply_features(gender_features_basic, devtest_set)

baseline_classifier = NaiveBayesClassifier.train(train_features_basic)
baseline_dev_accuracy = nltk.classify.accuracy(baseline_classifier, devtest_features_basic)

print(f"Baseline Naive Bayes dev-test accuracy: {baseline_dev_accuracy:.4f}")

Baseline Naive Bayes dev-test accuracy: 0.7540


The baseline accuracy is a useful starting point, but it leaves a lot of signal unused. Many names carry gender clues in more than just the final letter, so this gives a benchmark for improvement.

## Improved Feature Set

My first improvement adds the first letter, the final two letters, and the name length.

In [6]:
def gender_features_improved(name):
    name = name.lower()
    return {
        'first_letter': name[0],
        'last_letter': name[-1],
        'last_two': name[-2:] if len(name) > 1 else name,
        'length': len(name),
    }


print(gender_features_improved('John'))
print(gender_features_improved('Mary'))

{'first_letter': 'j', 'last_letter': 'n', 'last_two': 'hn', 'length': 4}
{'first_letter': 'm', 'last_letter': 'y', 'last_two': 'ry', 'length': 4}


In [7]:
train_features_improved = apply_features(gender_features_improved, train_set)
devtest_features_improved = apply_features(gender_features_improved, devtest_set)

improved_nb_classifier = NaiveBayesClassifier.train(train_features_improved)
improved_nb_dev_accuracy = nltk.classify.accuracy(improved_nb_classifier, devtest_features_improved)

print(f"Improved Naive Bayes dev-test accuracy: {improved_nb_dev_accuracy:.4f}")

Improved Naive Bayes dev-test accuracy: 0.7840


This feature set performs better than the baseline, so the extra information is helping. Short endings such as `-a`, `-ie`, or `-us` can be informative, but the first letter and overall length also seem to matter.

## Richer Features and Classifier Comparison

For a stronger model, I added a few more pattern-based features: the first two letters, the last three letters, the number of vowels, and whether the name ends with a vowel. I then compared two Chapter 6 classifiers on the same feature set.

In [8]:
def gender_features_richer(name):
    name = name.lower()
    return {
        'first_letter': name[0],
        'first_two': name[:2],
        'last_letter': name[-1],
        'last_two': name[-2:] if len(name) > 1 else name,
        'last_three': name[-3:] if len(name) > 2 else name,
        'length': len(name),
        'num_vowels': sum(ch in 'aeiou' for ch in name),
        'ends_with_vowel': name[-1] in 'aeiouy',
    }

In [9]:
train_features_richer = apply_features(gender_features_richer, train_set)
devtest_features_richer = apply_features(gender_features_richer, devtest_set)

richer_nb_classifier = NaiveBayesClassifier.train(train_features_richer)
richer_nb_dev_accuracy = nltk.classify.accuracy(richer_nb_classifier, devtest_features_richer)

richer_maxent_classifier = MaxentClassifier.train(
    train_features_richer,
    algorithm='iis',
    trace=0,
    max_iter=10,
)
richer_maxent_dev_accuracy = nltk.classify.accuracy(richer_maxent_classifier, devtest_features_richer)

print(f"Richer Naive Bayes dev-test accuracy: {richer_nb_dev_accuracy:.4f}")
print(f"Richer MaxEnt dev-test accuracy: {richer_maxent_dev_accuracy:.4f}")

Richer Naive Bayes dev-test accuracy: 0.8100
Richer MaxEnt dev-test accuracy: 0.8060


Both models improve on the earlier versions, and the richer Naive Bayes model does a little better than the richer MaxEnt model on the dev-test set. That gave me a good reason to keep pushing the feature engineering one step further.

## Final Feature Set

For the final round, I kept Naive Bayes and expanded the feature set further by adding short prefixes, short suffixes, vowel-start and vowel-end flags, and letter-presence/count features. Among the models shown here, this version gave the strongest dev-test result.

In [10]:
def gender_features_final(name):
    name = name.lower()
    features = {
        'length': len(name),
        'first_letter': name[0],
        'last_letter': name[-1],
        'first_two': name[:2],
        'last_two': name[-2:] if len(name) > 1 else name,
        'first_three': name[:3],
        'last_three': name[-3:] if len(name) > 2 else name,
        'starts_with_vowel': name[0] in 'aeiou',
        'ends_with_vowel': name[-1] in 'aeiouy',
        'num_vowels': sum(ch in 'aeiou' for ch in name),
    }

    for letter in 'abcdefghijklmnopqrstuvwxyz':
        features[f'has({letter})'] = letter in name
        features[f'count({letter})'] = name.count(letter)

    return features

In [11]:
train_features_final = apply_features(gender_features_final, train_set)
devtest_features_final = apply_features(gender_features_final, devtest_set)
test_features_final = apply_features(gender_features_final, test_set)

final_classifier = NaiveBayesClassifier.train(train_features_final)
final_dev_accuracy = nltk.classify.accuracy(final_classifier, devtest_features_final)
final_test_accuracy = nltk.classify.accuracy(final_classifier, test_features_final)

print(f"Final Naive Bayes dev-test accuracy: {final_dev_accuracy:.4f}")
print(f"Final Naive Bayes test accuracy: {final_test_accuracy:.4f}")

Final Naive Bayes dev-test accuracy: 0.8320
Final Naive Bayes test accuracy: 0.7860


This final model gives the best dev-test accuracy among the models in the notebook, so it is the fairest choice for the held-out test evaluation. The larger feature set seems to capture useful spelling patterns, especially prefixes, suffixes, and repeated letters.

## Summary of Development Results

In [12]:
results = [
    ('Baseline Naive Bayes', baseline_dev_accuracy),
    ('Improved Naive Bayes', improved_nb_dev_accuracy),
    ('Richer Naive Bayes', richer_nb_dev_accuracy),
    ('Richer MaxEnt', richer_maxent_dev_accuracy),
    ('Final Naive Bayes', final_dev_accuracy),
]

for model_name, score in results:
    print(f"{model_name}: {score:.4f}")

Baseline Naive Bayes: 0.7540
Improved Naive Bayes: 0.7840
Richer Naive Bayes: 0.8100
Richer MaxEnt: 0.8060
Final Naive Bayes: 0.8320


The overall pattern is clear: each richer feature set improves on the simple last-letter baseline. In this task, combining several small spelling cues works better than relying on any single clue.

## Comparison of Dev-Test and Test Performance

The final classifier scored `0.8320` on the dev-test set and `0.7860` on the test set.

The test score is lower than the dev-test score, and that is what I would expect. I used the dev-test set to guide model selection, so it is natural for performance there to be a little optimistic. The held-out test set gives a better estimate of how well the classifier generalizes to unseen names.